In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
from supabase import create_client, Client

In [2]:
load_dotenv()
URL = os.getenv("SUPABASE_URL")
BUCKET_NAME = os.getenv("SUPABASE_BUCKET")
KEY = os.getenv("SUPABASE_KEY")
supabase: Client = create_client(URL, KEY)

In [3]:
cloud_path = f"{URL}/storage/v1/object/public/{BUCKET_NAME}/review_ikd_raw.csv"
print(f"Menarik data dari awan: {cloud_path}")

df = pd.read_csv(cloud_path)
print(f'Total raw data: {len(df)} rows')

Menarik data dari awan: https://ffygtotdlnxhscukpmpu.supabase.co/storage/v1/object/public/dataset-ikd/review_ikd_raw.csv
Total raw data: 36679 rows


In [4]:
df_clean = df[df['rating'] != 3].copy()
print(f"\nRating distribution: \n{df_clean['rating'].value_counts().sort_index()}")


Rating distribution: 
rating
1    18332
2     2299
4     1137
5    13161
Name: count, dtype: int64


In [5]:
def set_label(score):
    if score <= 2:
        return 0
    else:
        return 1

df_clean['label'] = df_clean['rating'].apply(set_label)

In [6]:
print('--- RESULT ---')
print(f'Total Cleaned Rows: {len(df_clean)} rows')
print(f'Deleted rows (neutral): {len(df) - len(df_clean)} rows')

--- RESULT ---
Total Cleaned Rows: 34929 rows
Deleted rows (neutral): 1750 rows


In [7]:
label_counts = df_clean['label'].value_counts()
label_proportions = df_clean['label'].value_counts(normalize=True) * 100

In [8]:
print("\nDistribusi Label:")

stats = pd.DataFrame({
    'Jumlah': label_counts,
    'Persentase (%)': label_proportions.round(2)
})

stats.index = ['Negatif (0)', 'Positif (1)'] 
print(stats)


Distribusi Label:
             Jumlah  Persentase (%)
Negatif (0)   20631           59.07
Positif (1)   14298           40.93


In [9]:
newfile = 'ulasan_ikd_clean.csv'
newpath = f'./Result/{newfile}'
df_clean = df_clean[['review_text', 'label', 'date', 'app_version']]
df_clean.to_csv(newpath, index=False)

# Push Hasil Labeling ke Cloud
print("Mengunggah data berlabel ke Supabase...")
with open(newpath, 'rb') as f:
    supabase.storage.from_(BUCKET_NAME).upload(
        file=f, path=newfile, file_options={"content-type": "text/csv", "x-upsert": "true"}
    )
print("✅ Tahap 2 Selesai: Data Bersih siap masuk ke tahap Preprocessing!")
display(df_clean.head())

Mengunggah data berlabel ke Supabase...


✅ Tahap 2 Selesai: Data Bersih siap masuk ke tahap Preprocessing!


,review_text,label,date,app_version
0,sangat bagus,1,2026-01-06 18:59:47,2.0.4
1,"ambil foto diri error, loading lama dan lemot",0,2026-01-06 15:33:05,2.0.4
2,aplikasi bikin stressss...mau masuk ribet lg.....,0,2026-01-06 14:51:18,2.0.4
3,"pelayannya g ada, saya udah coba ajukan ulang ...",1,2026-01-06 14:40:48,2.0.4
4,maksud tujuan nya bagus. tapi kebanyakan aplik...,0,2026-01-06 13:30:05,2.0.4
